# Navier-Stokes equations — implementation notes

The incompressible Navier-Stokes equations form a coupled PDE system:

<img src="../static/1.png" height="120" style="display:block; margin:auto;">

where the stress tensor $\sigma(u, p)$ is:

<img src="../static/2.png" height="60" style="display:block; margin:auto;">

and the strain-rate tensor $\epsilon(u)$ is:

<img src="../static/3.png" height="60" style="display:block; margin:auto;">

## Splitting scheme (IPCS)

Rather than solving the coupled system directly (which has a saddle-point structure), 
we use the incremental pressure correction scheme (IPCS) — a modified Chorin splitting 
method that solves three linear problems per timestep. Full derivation: 
[Dokken (2023)](https://jsdokken.com/dolfinx-tutorial/chapter2/navierstokes.html).

### Step 1 — tentative velocity

<img src="../static/4.png" height="60" style="display:block; margin:auto;">

where:
- $\langle \cdot, \cdot \rangle$ — $L^2$ inner product over $\Omega$, i.e. $\int_\Omega uv \ dx$
- $\langle \cdot, \cdot \rangle_{\partial\Omega}$ — boundary inner product, i.e. $\int_{\partial\Omega} uv \ ds$
- $u^{n+1/2} \approx \frac{u^n + u^*}{2}$ — midpoint approximation
- $u^*$ — tentative velocity using pressure $p^n$ from previous timestep

### Step 2 — pressure correction

With $q$ the pressure test function and $v$ the velocity test function:

<img src="../static/5.png" height="80" style="display:block; margin:auto;">

which gives the variational form:

<img src="../static/6.png" height="80" style="display:block; margin:auto;">

In [ ]:
# Channel flow (Poiseyuille flow)

from mpi4py import MPI
from petsc4py import PETSc
import numpy as np
import pyvista

from dolfinx.fem import (
    Constant,
    Function,
    extract_function_spaces,
    functionspace,
    assemble_scalar,
    dirichletbc,
    form,
    locate_dofs_geometrical,
)

from dolfinx.fem.petsc import (
    assemble_matrix,
    assemble_vector,
    apply_lifting,
    create_vector,
    set_bc,
)

from dolfinx.io import VTXWriter
from dolfinx.mesh import create_unit_square
from dolfinx.plot import vtk_mesh
from basix.ufl import element
from ufl import (
    FacetNormal,
    Identity,
    TestFunction,
    TrialFunction,
    div,
    dot,
    ds,
    dx,
    inner,
    lhs,
    nabla_grad,
    rhs,
    sym,
)

mesh = create_unit_square(MPI.COMM_WORLD, 10, 10)
t = 0
T = 10
num_step = 500
dt = (T - t) / num_step

v_cg2 = element('Lagrange', mesh.basix_cell(), 2, shape = (mesh.geometry.dim,))
s_cg1 = element('Lagrange', mesh.basix_cell(), 1) 
V = functionspace(mesh, v_cg2)
Q = functionspace(mesh, s_cg1)

u = TrialFunction(V)
v = TestFunction(V)
p = TrialFunction(Q)
q = TestFunction(Q)

def walls(x):
    return np.logical_or( np.isclose(x[1], 0), np.isclose(x[1], 1))

wall_dofs = locate_dofs_geometrical(V, walls)
n_noslip = np.array((0,) * mesh.geometry.dim, dtype = PETSc.ScalarType)
bc_noslipe = dirichletbc(n_noslip, wall_dofs, V)

def inflow(x):
    return np.isclose(x[0],0)

ifnlow_dofs = locate_dofs_geometrical(Q, inflow)
bc_inflows = dirichletbc(PETSc.ScalarType(8), ifnlow_dofs, Q)

def outflow(x):
    return np.isclose(x[0], 1)

outflow_dofs = locate_dofs_geometrical(Q, outflow)
bc_outflow = dirichletbc(PETSc.ScalarType(0), outflow_dofs, Q)
bcu = [bc_noslipe]
bcp = [bc_inflows, bc_outflow]

u_n = Function(V)
u_n.name = "u_n"
U = 0.5 * (u_n + u_n)
n = FacetNormal(mesh)
f = Constant(mesh, PETSc.ScalarType((0,0)))
k = Constant(mesh, PETSc.ScalarType(dt))
mu = Constant(mesh, PETSc.ScalarType(1))
rho = Constant(mesh, PETSc.ScalarType(1))

def epsilon(u):
    """Strain rate tensor."""
    return sym(nabla_grad(u))


def sigma(u, p):
    """Stress tensor."""
    return 2 * mu * epsilon(u) - p * Identity(len(u))

p_n = Function(Q)
p_n.name = "p_n"
F1 = rho * dot((u - u_n) / k, v) * dx
F1 += rho * dot(dot(u_n, nabla_grad(u_n)), v) * dx
F1 += inner(sigma(U, p_n), epsilon(v)) * dx
F1 += dot(p_n * n, v) * ds - dot(mu * nabla_grad(U) * n, v) * ds
F1 -= dot(f, v) * dx
a1 = form(lhs(F1))
L1 = form(rhs(F1))

A1  = assemble_matrix(a1, bcs = bcu)
A1.assemble()
b1 = create_vector(extract_function_spaces(L1))

# Define variational problem for step 2
u_ = Function(V)
a2 = form(dot(nabla_grad(p), nabla_grad(q)) * dx)
L2 = form(dot(nabla_grad(p_n), nabla_grad(q)) * dx - (rho / k) * div(u_) * q * dx)
A2 = assemble_matrix(a2, bcs=bcp)
A2.assemble()
b2 = create_vector(extract_function_spaces(L2))

# Define variational problem for step 3
p_ = Function(Q)
a3 = form(rho * dot(u, v) * dx)
L3 = form(rho * dot(u_, v) * dx - k * dot(nabla_grad(p_ - p_n), v) * dx)
A3 = assemble_matrix(a3)
A3.assemble()
b3 = create_vector(extract_function_spaces(L3))


# SOLVERS 
solver = 